> 在线讲义：[第 23 章 · MDP 与 Bellman 的完整推导：值函数、最优方程、压缩映射与值迭代](https://wikieden.github.io/math-and-eval-stats/lessons/23-mdp-bellman.html) · 本书《数学与实验统计》

# 第 23 章 · MDP 与 Bellman 的完整推导：值函数、最优方程、压缩映射与值迭代

配套讲义：仓库里 `math-and-eval-stats/notes/lessons/23-mdp-bellman.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy / SciPy（Colab 自带），不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第一步 期望方程 | (I − γP)⁻¹r | `policy_value(pi)` | 一直走：[6.2, 8.0, 10.0] |
| 第二步 值迭代 | V ← TV | `value_iteration()` | 176 次；变化 1, 0.9, 0.81, … |
| 第三步 压缩 | 比值 ≤ γ | `contraction_check()` | 最大 0.9000 |
| 第四步 次数 | log 公式 | `iterations_needed(1e-6, 0.99, 10)` | 1604 |
| 第五步 策略迭代 | 评估 + 贪心 | `policy_iteration()` | 3 轮 |

例子：一个 3 个状态、2 个动作的 MDP（编的）：状态 0 → 1 → 2 是一条链；在 0、1 上"走"往前一格、扣 1 分，"停"原地不动、0 分；
到了 2 以后，不管做什么都留在 2、每步 +1（无限时域，V*(2) = 1/(1 − γ) = 10）。
里程碑 ME-M19 的 4×4 网格收敛曲线留给自己做。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import numpy as np
GAMMA = 0.9
P = np.array([[[1, 0, 0], [0, 1, 0], [0, 0, 1]],
              [[0, 1, 0], [0, 0, 1], [0, 0, 1]]], float)
R = np.array([[0.0, 0.0, 1.0],
              [-1.0, -1.0, 1.0]])

**函数 `policy_value`**：Bellman 期望方程：V^π = r_π + γ P_π V^π → V^π = (I − γP_π)⁻¹ r_π。pi[s] 是在 s 选的动作。

In [ ]:
def policy_value(pi: np.ndarray, gamma: float = GAMMA) -> np.ndarray:
    """Bellman 期望方程：V^π = r_π + γ P_π V^π → V^π = (I − γP_π)⁻¹ r_π。pi[s] 是在 s 选的动作。"""
    p = np.array([P[pi[s], s] for s in range(3)])
    r = np.array([R[pi[s], s] for s in range(3)])
    return np.linalg.solve(np.eye(3) - gamma * p, r)

**函数 `bellman_optimal`**：Bellman 最优算子：(T V)(s) = max_a [R(a, s) + γ Σ P(a, s, s') V(s')]。

In [ ]:
def bellman_optimal(v: np.ndarray, gamma: float = GAMMA) -> np.ndarray:
    """Bellman 最优算子：(T V)(s) = max_a [R(a, s) + γ Σ P(a, s, s') V(s')]。"""
    return np.max(R + gamma * P @ v, axis=0)

**函数 `value_iteration`**：反复套最优算子，返回 (V*, 迭代次数, 每次的 ‖V_{k+1} − V_k‖∞)。

In [ ]:
def value_iteration(tol: float = 1e-8, gamma: float = GAMMA) -> tuple[np.ndarray, int, list[float]]:
    """反复套最优算子，返回 (V*, 迭代次数, 每次的 ‖V_{k+1} − V_k‖∞)。"""
    v, gaps = np.zeros(3), []
    while True:
        nv = bellman_optimal(v, gamma)
        gaps.append(float(np.abs(nv - v).max()))
        v = nv
        if gaps[-1] < tol:
            return v, len(gaps), gaps

**函数 `contraction_check`**：随便取两个 V、U，看 ‖TV − TU‖∞ / ‖V − U‖∞ 是不是 ≤ γ。

In [ ]:
def contraction_check(seed: int = 0, gamma: float = GAMMA) -> float:
    """随便取两个 V、U，看 ‖TV − TU‖∞ / ‖V − U‖∞ 是不是 ≤ γ。"""
    rng = np.random.default_rng(seed)
    worst = 0.0
    for _ in range(1000):
        v, u = rng.normal(0, 10, 3), rng.normal(0, 10, 3)
        worst = max(worst, np.abs(bellman_optimal(v, gamma) - bellman_optimal(u, gamma)).max() / np.abs(v - u).max())
    return worst

**函数 `iterations_needed`**：误差每次至少乘 γ：要从 v0_gap 降到 eps，至少 ⌈log(eps/v0_gap)/log γ⌉ 次。

In [ ]:
def iterations_needed(eps: float, gamma: float, v0_gap: float) -> int:
    """误差每次至少乘 γ：要从 v0_gap 降到 eps，至少 ⌈log(eps/v0_gap)/log γ⌉ 次。"""
    return int(np.ceil(np.log(eps / v0_gap) / np.log(gamma)))

**函数 `greedy`**：按 V 贪心取动作。

In [ ]:
def greedy(v: np.ndarray, gamma: float = GAMMA) -> np.ndarray:
    """按 V 贪心取动作。"""
    return np.argmax(R + gamma * P @ v, axis=0)

**函数 `policy_iteration`**：策略评估（解线性方程）+ 贪心改进，直到策略不变。

In [ ]:
def policy_iteration(gamma: float = GAMMA) -> tuple[np.ndarray, np.ndarray, int]:
    """策略评估（解线性方程）+ 贪心改进，直到策略不变。"""
    pi, k = np.zeros(3, int), 0
    while True:
        v = policy_value(pi, gamma)
        new = greedy(v, gamma)
        k += 1
        if np.array_equal(new, pi):
            return pi, v, k
        pi = new

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 Bellman 期望方程：固定策略，值函数是一个线性方程组的解")
for name, pi in (("一直停", np.array([0, 0, 0])), ("一直走", np.array([1, 1, 0]))):
    print(f"  {name}：V = {np.round(policy_value(pi), 4).tolist()}")

In [ ]:
print("\n第二步 Bellman 最优方程与值迭代")
v, n, gaps = value_iteration()
print(f"  V* = {np.round(v, 4).tolist()}，{n} 次收敛；最优策略 {greedy(v).tolist()}（0 停 / 1 走）")
print(f"  前几次的 ‖V_(k+1) − V_k‖∞：{[round(g, 4) for g in gaps[:5]]}")

In [ ]:
print("\n第三步 压缩映射：‖TV − TU‖∞ ≤ γ‖V − U‖∞")
print(f"  随机 1000 对 V、U，比值最大 {contraction_check():.4f}（γ = {GAMMA}）")

In [ ]:
print("\n第四步 要多少次迭代：误差每次至少乘 γ")
for g in (0.9, 0.99):
    print(f"  γ = {g}：初始误差 10 降到 1e-6 至少要 {iterations_needed(1e-6, g, 10.0)} 次")

In [ ]:
print("\n第五步 策略迭代：评估 + 贪心改进，几步就停")
pi, v, k = policy_iteration()
print(f"  {k} 轮策略迭代得到 {pi.tolist()}，V = {np.round(v, 4).tolist()}")

## 3. 代码里才看得到的

1. **P 的形状是 (动作, 状态, 下一状态)**：`P @ V` 对最后一维求和，得到 (动作, 状态) 的期望下一步值，再加 R 就是 Q 表。
2. **在状态 2 两个动作打平**：`argmax` 返回第一个（停），所以最优策略打印成 [1, 1, 0]；打平时任何一个都是最优。
3. **值迭代的停止条件用相邻两次的差**：真正的误差 ‖V_k − V*‖ 不知道（不知道 V*），但 ‖V_{k+1} − V_k‖ ≤ ε 时，‖V_{k+1} − V*‖ ≤ γε/(1 − γ)。

## 4. 自己改一改

1. 把 γ 改成 0.5，V* 变成多少？最优策略还是"走"吗？
2. 给链加一个状态 3（0 → 1 → 2 → 3，3 才有 +1），值迭代要多少次？奖励往回传要几轮才到状态 0？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 存折的价值等于本月利息加打折的下月，期望方程解一个线性方程组；
> 最优方程带 max，值迭代反复套；
> 算子是 γ 压缩，不动点唯一，从哪出发都收敛；
> 迭代次数约 log 除 1 减 γ；策略迭代评估加贪心，几轮就停。